# Experiment 5: Model Selection
In this final experiment of the series, we compare various machine learning algorithms to determine which one performs best for YouTube comment sentiment analysis.

### Models to compare:
1. **Logistic Regression (LoR)**: Linear model, fast and often a strong baseline for text.
2. **Naive Bayes (MultinomialNB)**: Probabilistic model, extremely fast and effective for text.
3. **Random Forest (RF)**: Ensemble of decision trees.
4. **XGBoost & LightGBM**: Gradient Boosted Decision Trees, state-of-the-art for many tabular/structured tasks.
5. **Support Vector Machine (SVM)**: Effective in high-dimensional spaces (like text).
6. **K-Nearest Neighbors (KNN)**: Simple distance-based model.

### Why compare models?
Different algorithms have different biases. Some handle high-dimensional sparse data (like text) better, while others are better at capturing non-linear relationships.

In [2]:
import pandas as pd
import numpy as np
import mlflow
import os
import tempfile
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                              recall_score, classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_sample_weight
from scipy.sparse import hstack
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
try:
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
except ImportError:
    print("XGBoost or LightGBM not installed. Skipping those.")



In [3]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Model Selection - 2")

2026/04/14 11:53:27 INFO mlflow.tracking.fluent: Experiment with name 'Model Selection - 2' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/16', creation_time=1776147807218, experiment_id='16', last_update_time=1776147807218, lifecycle_stage='active', name='Model Selection - 2', tags={}, workspace='default'>

In [4]:
# Load and preprocess
data = pd.read_csv(r'../data/processed/dataset.csv')
data = data.dropna(subset=['text_processed', 'sentiment']).drop_duplicates()

X_text = data['text_processed']
X_numeric = data[['char_count', 'word_count', 'avg_word_len']].values
y = data['sentiment']

# Label encoding for y if needed by some models (like XGBoost)
from sklearn.preprocessing import LabelEncoder, RobustScaler
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_numeric, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Text + Scaling setup
tfidf = TfidfVectorizer(max_features=2500, ngram_range=(1,1))
scaler = RobustScaler()

X_text_train_vec = tfidf.fit_transform(X_text_train)
X_text_test_vec = tfidf.transform(X_text_test)
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)

X_train_final = hstack([X_text_train_vec, X_num_train_scaled])
X_test_final = hstack([X_text_test_vec, X_num_test_scaled])

# Special data for Naive Bayes (No negative values)
nb_scaler = MinMaxScaler()
X_num_train_nb = nb_scaler.fit_transform(X_num_train)
X_num_test_nb = nb_scaler.transform(X_num_test)
X_train_nb = hstack([X_text_train_vec, X_num_train_nb])
X_test_nb = hstack([X_text_test_vec, X_num_test_nb])

In [5]:
# Define the core models we want to evaluate
base_models = [
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=42)),
    ("NaiveBayes", MultinomialNB(alpha=1.0)),
    ("ComplementNB", ComplementNB()),
    ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("SVM", SVC(kernel='rbf', random_state=42, probability=True)),
    ("LinearSVC", LinearSVC(random_state=42, max_iter=2000)),
    ("KNN_k5", KNeighborsClassifier(n_neighbors=5)),
]

# Add Gradient Boosting
try:
    base_models.append(("LightGBM", LGBMClassifier(random_state=42, verbosity=-1)))
    base_models.append(("XGBoost", XGBClassifier(random_state=42, eval_metric='mlogloss')))
except:
    pass

# Define the three distinct scenarios requested:
# 1. Baseline (No resampling, No weights)
# 2. Class Weights (Only for supported models, No resampling)
# 3. Resampling (SMOTE variants, No weights)
strategies = ["Baseline", "ClassWeights", "SMOTE", "SMOTETomek", "BorderlineSMOTE", "ADASYN"]

for model_name, base_clf in base_models:
    for strategy in strategies:
        
        # ── Logic Filtering ──
        has_weight_support = hasattr(base_clf, "class_weight") or model_name == "XGBoost"
        
        # 1. Skip ClassWeights for models that don't support it (KNN/NB)
        if strategy == "ClassWeights" and not has_weight_support:
            continue
            
        # 2. Keep BorderlineSMOTE and ADASYN specialized for SVM (as per previous industry request)
        # or apply them generally if desired. Here we keep them for SVM to avoid massive run times.
        if strategy in ["BorderlineSMOTE", "ADASYN"] and model_name != "SVM":
            continue
            
        run_name = f"{model_name}_{strategy}"
        
        with mlflow.start_run(run_name=run_name):
            # Setup Training Data (NB variants use MinMaxScaler)
            is_nb = "NB" in model_name or "Naive" in model_name
            X_tr = X_train_nb if is_nb else X_train_final
            X_ts = X_test_nb if is_nb else X_test_final
            
            # Configure the classifier instance
            # We must use a fresh clone/instance to avoid parameter bleed between runs
            from sklearn.base import clone
            curr_clf = clone(base_clf)
            
            # Scenario A: Class Weights Only
            if strategy == "ClassWeights":
                if model_name != "XGBoost":
                    curr_clf.set_params(class_weight='balanced')
            
            # Scenario B: Resampling Only (Ensure weights are None/Default)
            resampler = None
            if strategy == "SMOTE":
                resampler = SMOTE(random_state=42)
            elif strategy == "SMOTETomek":
                resampler = SMOTETomek(random_state=42)
            elif strategy == "BorderlineSMOTE":
                resampler = BorderlineSMOTE(random_state=42)
            elif strategy == "ADASYN":
                resampler = ADASYN(random_state=42)

            if resampler:
                # Industry standard: wrap in pipeline to prevent leakage
                model = Pipeline([('resample', resampler), ('clf', curr_clf)])
            else:
                model = curr_clf

            # ── Train ─────────────────────────────────────────────────────────────────
            if model_name == "XGBoost" and strategy == "ClassWeights":
                # Apply weights manually for XGBoost in .fit()
                sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
                model.fit(X_tr, y_train, sample_weight=sample_weights)
            else:
                model.fit(X_tr, y_train)
                
            y_pred = model.predict(X_ts)
            
            # ── Log Parameters ────────────────────────────────────────────────────────
            params = {
                "test_size"             : 0.2,
                "stratify"              : True,
                "random_state"          : 42,
                "representation"        : "TFIDF_500",
                "scaler"                : "MinMaxScaler" if is_nb else "RobustScalar",
                "model_name"            : model_name,
                "strategy_category"     : "Resampling" if resampler else strategy,
                "specific_strategy"     : strategy,
                "weights_applied"       : "balanced" if strategy == "ClassWeights" else "None",
            }
            
            if hasattr(curr_clf, 'get_params'):
                model_params = curr_clf.get_params()
                for k in ['C', 'alpha', 'n_estimators', 'max_depth', 'n_neighbors', 'kernel']:
                    if k in model_params:
                        params[f"model_{k}"] = str(model_params[k])
            
            mlflow.log_params(params)

            # ── Log Metrics ───────────────────────────────────────────────────────────
            report_dict:dict = classification_report(y_test, y_pred, output_dict=True) # type: ignore
            
            metrics = {
                "accuracy"         : accuracy_score(y_test, y_pred),
                "f1_weighted"      : report_dict['weighted avg']['f1-score'],
                "f1_macro"         : report_dict['macro avg']['f1-score'],
                "precision_weighted": report_dict['weighted avg']['precision'],
                "precision_macro"  : report_dict['macro avg']['precision'],
                "recall_weighted"  : report_dict['weighted avg']['recall'],
                "recall_macro"     : report_dict['macro avg']['recall'],
            }
            
            # Log Individual Class Metrics
            for i, label in enumerate(le.classes_):
                metrics[f"f1_class_{label}"] = report_dict[str(i)]['f1-score']
                metrics[f"precision_class_{label}"] = report_dict[str(i)]['precision']
                metrics[f"recall_class_{label}"] = report_dict[str(i)]['recall']
                
            mlflow.log_metrics(metrics)

            # ── Log Artifacts ─────────────────────────────────────────────────────────
            report_str = classification_report(y_test, y_pred, target_names=le.classes_.astype(str))
            print(f"Finished: {run_name} | F1-Macro: {metrics['f1_macro']:.4f}")

            with tempfile.TemporaryDirectory() as tmp_dir:
                report_path = os.path.join(tmp_dir, "classification_report.txt")
                with open(report_path, "w") as f:
                    f.write(str(report_str))
                mlflow.log_artifact(report_path)

                plt.figure(figsize=(8, 5))
                sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
                            xticklabels=le.classes_, yticklabels=le.classes_)
                plt.title(f'Confusion Matrix - {run_name}')
                plt.tight_layout()
                
                cm_path = os.path.join(tmp_dir, "confusion_matrix.png")
                plt.savefig(cm_path)
                mlflow.log_artifact(cm_path)
                plt.close()


Finished: LogisticRegression_Baseline | F1-Macro: 0.8763
🏃 View run LogisticRegression_Baseline at: http://localhost:5000/#/experiments/16/runs/8e4b47e7e355478da3796ea8a41f470f
🧪 View experiment at: http://localhost:5000/#/experiments/16
Finished: LogisticRegression_ClassWeights | F1-Macro: 0.8804
🏃 View run LogisticRegression_ClassWeights at: http://localhost:5000/#/experiments/16/runs/fe731ea633064ab38136bb695d9d888a
🧪 View experiment at: http://localhost:5000/#/experiments/16
Finished: LogisticRegression_SMOTE | F1-Macro: 0.8789
🏃 View run LogisticRegression_SMOTE at: http://localhost:5000/#/experiments/16/runs/fb839be003f5487f954b0ab976cd21f4
🧪 View experiment at: http://localhost:5000/#/experiments/16
Finished: LogisticRegression_SMOTETomek | F1-Macro: 0.8790
🏃 View run LogisticRegression_SMOTETomek at: http://localhost:5000/#/experiments/16/runs/0a54e5c0b5534f238c9d3e2040f6d74a
🧪 View experiment at: http://localhost:5000/#/experiments/16
Finished: NaiveBayes_Baseline | F1-Macro: 

c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Finished: LightGBM_Baseline | F1-Macro: 0.8665
🏃 View run LightGBM_Baseline at: http://localhost:5000/#/experiments/16/runs/cb23401b941346939b3fb4840956c393
🧪 View experiment at: http://localhost:5000/#/experiments/16


c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Finished: LightGBM_ClassWeights | F1-Macro: 0.8620
🏃 View run LightGBM_ClassWeights at: http://localhost:5000/#/experiments/16/runs/8d46266b5a8f4f76a1e1e53d5fd135bc
🧪 View experiment at: http://localhost:5000/#/experiments/16


c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Finished: LightGBM_SMOTE | F1-Macro: 0.8659
🏃 View run LightGBM_SMOTE at: http://localhost:5000/#/experiments/16/runs/c708d1a07fb04d4fbfe2dfd62ec1d892
🧪 View experiment at: http://localhost:5000/#/experiments/16


c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\data-science-projects\major-projects\youtube_comment_analysis\.venv\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Finished: LightGBM_SMOTETomek | F1-Macro: 0.8712
🏃 View run LightGBM_SMOTETomek at: http://localhost:5000/#/experiments/16/runs/218cc526eb0640ae94b2f02f50377222
🧪 View experiment at: http://localhost:5000/#/experiments/16
Finished: XGBoost_Baseline | F1-Macro: 0.8566
🏃 View run XGBoost_Baseline at: http://localhost:5000/#/experiments/16/runs/1f1d2d90dbf647a2a0bcbb673272f01b
🧪 View experiment at: http://localhost:5000/#/experiments/16
Finished: XGBoost_ClassWeights | F1-Macro: 0.8444
🏃 View run XGBoost_ClassWeights at: http://localhost:5000/#/experiments/16/runs/77850bc4a99c47f099bb8121762ecd82
🧪 View experiment at: http://localhost:5000/#/experiments/16
Finished: XGBoost_SMOTE | F1-Macro: 0.8570
🏃 View run XGBoost_SMOTE at: http://localhost:5000/#/experiments/16/runs/dcf8d3047b2445f0bb0218a656804474
🧪 View experiment at: http://localhost:5000/#/experiments/16
Finished: XGBoost_SMOTETomek | F1-Macro: 0.8556
🏃 View run XGBoost_SMOTETomek at: http://localhost:5000/#/experiments/16/runs/1d

Conculusion - LightGBM (Tree Based) w class weight and Logistic Regression (Linear Model) - baseline are doing good.